# Clustering (et exploitation des données géographiques)

Dans cette partie nous exploitons les données géographiques et réalisons le clustering. 

Nous exportons des statistiques par commune pour croisement avec les données de vote. 

In [ ]:
from BDTopo_fonctions import load_gpkg
gdf=load_gpkg("Sitadel/df_clustering_fulldep_1000m3.gpkg")

Téléchargement depuis mgarbe/Sitadel/df_clustering_fulldep_1000m3.gpkg ...
Chargement réussi (716878 lignes)


In [ ]:
import importlib
import clustering_fonctions
importlib.reload(clustering_fonctions)
from clustering_fonctions import run_dbscan_parallele
temp=run_dbscan_parallele(gdf,700,5)

In [ ]:
# CLUSTER TOTAL, TOUS BAT EXISTANTS OU AUTORISES, ANNEE PAR ANNEE 
import pandas as pd
temp_sorted = temp.sort_values("Annee_REF")
results = []
for annee in sorted(temp_sorted["Annee_REF"].unique()):
    mask = temp_sorted["Annee_REF"] <= annee  # toutes les années <= annee
    cluster_col = temp_sorted.loc[mask, "cluster_id"]
    pct_cluster = (cluster_col > 0).sum() / len(cluster_col) * 100
    results.append({
        "Annee": annee,
        "pct_cluster_cumule": pct_cluster
    })
df_cumule = pd.DataFrame(results)
print(df_cumule)

    Annee  pct_cluster_cumule
0    2013           81.718611
1    2014           81.571462
2    2015           81.386664
3    2016           81.213144
4    2017           81.116015
5    2018           81.070397
6    2019           81.715458
7    2020           81.749589
8    2021           81.779282
9    2022           81.824865
10   2023           81.846602
11   2024           81.859992
12   2025           81.871392


In [ ]:
#PC QUI EN 2025 SONT DANS UN CLUSTER
temp[temp["Base"]=="Sitadel"].groupby("Annee_REF")["cluster_id"].apply(lambda x: (x > 0).sum() / len(x) * 100)

Annee_REF
2014    89.628757
2015    89.614075
2016    89.897904
2017    90.608993
2018    91.860465
2019    91.026429
2020    91.699295
2021    93.204530
2022    92.597240
2023    91.736164
2024    90.120275
2025    90.310559
Name: cluster_id, dtype: float64

In [ ]:
#PC QUI SONT DANS UN CLUSTER AU MOMENT DE AUTORISATION 

temp_sitadel = temp[(temp["Base"] == "Sitadel")]

for annee in sorted(temp_sitadel["Annee_REF"].unique()):
    col = f"cluster_id_{annee}"
    mask = temp_sitadel["Annee_REF"] == annee
    cluster_col = temp_sitadel.loc[mask, col]
    pct_cluster = (cluster_col > 0).sum() / len(cluster_col) * 100
    print(f"Année {annee}: {pct_cluster:.1f}% des bâtiments clusterisés")

Année 2014: 86.3% des bâtiments clusterisés
Année 2015: 86.8% des bâtiments clusterisés
Année 2016: 88.1% des bâtiments clusterisés
Année 2017: 88.8% des bâtiments clusterisés
Année 2018: 89.9% des bâtiments clusterisés
Année 2019: 89.6% des bâtiments clusterisés
Année 2020: 90.9% des bâtiments clusterisés
Année 2021: 91.9% des bâtiments clusterisés
Année 2022: 92.0% des bâtiments clusterisés
Année 2023: 91.3% des bâtiments clusterisés
Année 2024: 89.9% des bâtiments clusterisés
Année 2025: 90.3% des bâtiments clusterisés


In [ ]:
annees = sorted(temp["Annee_REF"].unique())
result_list = []
for annee in annees:
    col = f"cluster_id_{annee}"
    if col in temp.columns:
        # compter les clusters uniques non-négatifs
        nb_clusters = temp[col][temp[col] > 0].nunique()
        result_list.append({"Annee_REF": annee, "nb_clusters": nb_clusters})

df_clusters_par_annee = pd.DataFrame(result_list)
print(df_clusters_par_annee)

    Annee_REF  nb_clusters
0        2013        14643
1        2014        14806
2        2015        14933
3        2016        15052
4        2017        15146
5        2018        15206
6        2019        15559
7        2020        15615
8        2021        15635
9        2022        15692
10       2023        15692
11       2024        15748
12       2025        15777
